# 24. backtrader 事件驱动回测

## 学习目标

通过本次学习，你将能够：

1. **理解事件驱动 vs 向量化回测的区别**
2. **掌握 backtrader 的基本架构**
3. **理解订单类型与撮合逻辑**
4. **实现双均线策略并添加手续费**
5. **对比扣费前后的策略表现**

## 知识地图

```
backtrader 事件驱动回测
├── 回测框架对比
│   ├── 向量化回测（pandas）
│   └── 事件驱动回测（backtrader）
├── backtrader 架构
│   ├── Cerebro（大脑）
│   ├── DataFeed（数据源）
│   ├── Strategy（策略）
│   ├── Broker（经纪人）
│   └── Analyzer（分析器）
├── 订单类型
│   ├── 市价单（Market Order）
│   ├── 限价单（Limit Order）
│   └── 止损单（Stop Order）
└── 交易成本
    ├── 手续费（Commission）
    └── 滑点（Slippage）
```

## 环境依赖

```bash
pip install backtrader matplotlib pandas
```

In [ ]:
import backtrader as bt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f"backtrader 版本: {bt.__version__}")
print("环境准备完成！")

---
## 1. 理论基础

### 1.1 事件驱动 vs 向量化回测

| 特性 | 向量化回测 | 事件驱动回测 |
|------|------------|--------------|
| **原理** | 一次性计算所有信号 | 逐根 K 线模拟 |
| **速度** | 快 | 慢 |
| **真实性** | 较低（可能有未来数据） | 高（严格时序） |
| **灵活性** | 低 | 高 |
| **适用场景** | 快速验证想法 | 详细回测、实盘对接 |
| **代表工具** | pandas + numpy | backtrader、vnpy |

### 1.2 为什么需要事件驱动？

向量化回测的问题：

1. **未来数据泄露**：容易不小心用到未来数据
2. **撮合逻辑简化**：无法精确模拟订单撮合
3. **交易成本粗糙**：难以精确计算手续费和滑点
4. **无法对接实盘**：回测和实盘逻辑不一致

事件驱动回测的优势：

1. **严格时序**：每根 K 线只用到之前的数据
2. **精确撮合**：模拟真实的订单撮合过程
3. **灵活成本**：可以精确设置手续费和滑点
4. **实盘对接**：回测逻辑可以直接用于实盘

### 1.3 backtrader 架构

```
Cerebro（大脑）
├── DataFeed（数据源）
│   └── PandasData / YahooFinanceData
├── Strategy（策略）
│   └── next() / buy() / sell()
├── Broker（经纪人）
│   └── 手续费 / 滑点 / 资金
├── Analyzer（分析器）
│   └── SharpeRatio / DrawDown / Returns
└── Observer（观察者）
    └── Cash / Value / Trades
```

---
## 2. 数据准备

In [ ]:
def generate_stock_data(n_days=500):
    """
    生成模拟股票数据
    """
    np.random.seed(42)
    
    dates = pd.date_range('2020-01-01', periods=n_days, freq='B')
    
    # 生成价格（带趋势和波动）
    returns = np.random.normal(0.0003, 0.02, n_days)
    # 添加动量效应
    for i in range(1, n_days):
        returns[i] += 0.1 * returns[i-1]
    
    close = 100 * np.exp(np.cumsum(returns))
    
    # 生成 OHLCV
    high = close * (1 + np.abs(np.random.normal(0, 0.01, n_days)))
    low = close * (1 - np.abs(np.random.normal(0, 0.01, n_days)))
    open_price = close * (1 + np.random.normal(0, 0.005, n_days))
    volume = np.random.randint(1000000, 10000000, n_days)
    
    df = pd.DataFrame({
        'open': open_price,
        'high': high,
        'low': low,
        'close': close,
        'volume': volume
    }, index=dates)
    
    return df


# 生成数据
data_df = generate_stock_data(500)

print(f"数据信息：")
print(f"  时间范围: {data_df.index[0]} 到 {data_df.index[-1]}")
print(f"  数据点数: {len(data_df)}")
print(f"  初始价格: {data_df['close'].iloc[0]:.2f}")
print(f"  最终价格: {data_df['close'].iloc[-1]:.2f}")

# 绘制价格
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(data_df.index, data_df['close'], linewidth=1.5)
ax.set_xlabel('日期')
ax.set_ylabel('价格')
ax.set_title('股票价格走势')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. backtrader 基础

### 3.1 创建 Cerebro

**Cerebro** 是 backtrader 的大脑，负责协调所有组件。

In [ ]:
# 创建 Cerebro
cerebro = bt.Cerebro()

# 设置初始资金
cerebro.broker.setcash(100000)

print(f"初始资金: {cerebro.broker.getvalue():.2f}")

### 3.2 添加数据源

backtrader 支持多种数据源，我们使用 pandas DataFrame。

In [ ]:
# 创建数据源
class PandasData(bt.feeds.PandasData):
    """
    自定义 pandas 数据源
    """
    params = (
        ('datetime', None),  # 使用 index 作为日期
        ('open', 'open'),
        ('high', 'high'),
        ('low', 'low'),
        ('close', 'close'),
        ('volume', 'volume'),
        ('openinterest', -1)  # 无持仓量
    )


# 添加数据到 Cerebro
data = PandasData(dataname=data_df)
cerebro.adddata(data)

print("数据源添加完成！")

### 3.3 定义策略

双均线策略：
- 短期均线上穿长期均线 → 买入
- 短期均线下穿长期均线 → 卖出

In [ ]:
class DualMovingAverage(bt.Strategy):
    """
    双均线策略
    """
    params = (
        ('fast_period', 10),   # 短期均线
        ('slow_period', 30),   # 长期均线
        ('printlog', False),   # 是否打印日志
    )
    
    def __init__(self):
        # 计算均线
        self.fast_ma = bt.indicators.SMA(
            self.data.close, period=self.params.fast_period
        )
        self.slow_ma = bt.indicators.SMA(
            self.data.close, period=self.params.slow_period
        )
        
        # 金叉死叉信号
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)
        
        # 记录交易
        self.trades = []
    
    def next(self):
        # 当前没有持仓
        if not self.position:
            # 金叉：买入
            if self.crossover > 0:
                self.buy()
        # 当前有持仓
        else:
            # 死叉：卖出
            if self.crossover < 0:
                self.sell()
    
    def notify_trade(self, trade):
        """记录交易"""
        if trade.isclosed:
            self.trades.append({
                'date': self.data.datetime.date(0),
                'pnl': trade.pnl,
                'pnlcomm': trade.pnlcomm
            })
            if self.params.printlog:
                print(f"交易完成: 日期={self.data.datetime.date(0)}, "
                      f"盈亏={trade.pnl:.2f}, 扣费后={trade.pnlcomm:.2f}")


print("策略定义完成！")

### 3.4 运行回测（无手续费）

In [ ]:
# 添加策略
cerebro.addstrategy(DualMovingAverage, printlog=False)

# 运行回测
print("运行回测（无手续费）...")
results = cerebro.run()
strategy = results[0]

# 获取结果
final_value = cerebro.broker.getvalue()
total_return = (final_value / 100000 - 1) * 100

print(f"\n回测结果（无手续费）：")
print(f"  初始资金: 100,000.00")
print(f"  最终资金: {final_value:,.2f}")
print(f"  总收益率: {total_return:.2f}%")
print(f"  交易次数: {len(strategy.trades)}")

# 绘制结果
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data_df.index, data_df['close'], label='价格', linewidth=1.5)
ax.plot(data_df.index, strategy.fast_ma.array[:len(data_df)], 
        label=f'MA{10}', linewidth=1, alpha=0.7)
ax.plot(data_df.index, strategy.slow_ma.array[:len(data_df)], 
        label=f'MA{30}', linewidth=1, alpha=0.7)
ax.set_xlabel('日期')
ax.set_ylabel('价格')
ax.set_title('双均线策略（无手续费）')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. 手续费与滑点

### 4.1 手续费设置

backtrader 支持多种手续费模式：

| 模式 | 说明 | 示例 |
|------|------|------|
| **百分比** | 按交易金额的百分比 | 0.1% |
| **固定金额** | 每笔交易固定费用 | 5 元 |
| **混合模式** | 百分比 + 固定金额 | 0.1% + 5 元 |

### 4.2 滑点设置

**滑点**是指实际成交价格与预期价格的差异。

- **产生原因**：市场波动、流动性不足
- **设置方式**：固定滑点或百分比滑点
- **影响**：降低策略收益

In [ ]:
# 创建新的 Cerebro（带手续费）
cerebro_with_cost = bt.Cerebro()

# 设置初始资金
cerebro_with_cost.broker.setcash(100000)

# 设置手续费：0.1%
cerebro_with_cost.broker.setcommission(commission=0.001)

# 设置滑点：0.05%
cerebro_with_cost.broker.set_slippage_perc(0.0005)

# 添加数据
data_with_cost = PandasData(dataname=data_df)
cerebro_with_cost.adddata(data_with_cost)

# 添加策略
cerebro_with_cost.addstrategy(DualMovingAverage, printlog=False)

# 运行回测
print("运行回测（含手续费 0.1% + 滑点 0.05%）...")
results_with_cost = cerebro_with_cost.run()
strategy_with_cost = results_with_cost[0]

# 获取结果
final_value_with_cost = cerebro_with_cost.broker.getvalue()
total_return_with_cost = (final_value_with_cost / 100000 - 1) * 100

print(f"\n回测结果（含手续费）：")
print(f"  初始资金: 100,000.00")
print(f"  最终资金: {final_value_with_cost:,.2f}")
print(f"  总收益率: {total_return_with_cost:.2f}%")
print(f"  交易次数: {len(strategy_with_cost.trades)}")

### 4.3 对比扣费前后表现

In [ ]:
# 计算交易成本
def calculate_trade_costs(trades):
    """
    计算交易成本
    """
    total_pnl = sum(t['pnl'] for t in trades)
    total_pnlcomm = sum(t['pnlcomm'] for t in trades)
    total_cost = total_pnl - total_pnlcomm
    return total_pnl, total_pnlcomm, total_cost


# 无手续费结果
pnl_no_cost, pnlcomm_no_cost, cost_no_cost = calculate_trade_costs(strategy.trades)

# 有手续费结果
pnl_with_cost, pnlcomm_with_cost, cost_with_cost = calculate_trade_costs(strategy_with_cost.trades)

print("=" * 60)
print("扣费前后对比")
print("=" * 60)

print(f"\n无手续费：")
print(f"  总盈亏: {pnl_no_cost:,.2f}")
print(f"  最终资金: {100000 + pnl_no_cost:,.2f}")
print(f"  总收益率: {(pnl_no_cost / 100000) * 100:.2f}%")

print(f"\n含手续费（0.1% + 滑点 0.05%）：")
print(f"  总盈亏（扣费前）: {pnl_with_cost:,.2f}")
print(f"  总盈亏（扣费后）: {pnlcomm_with_cost:,.2f}")
print(f"  交易成本: {cost_with_cost:,.2f}")
print(f"  最终资金: {100000 + pnlcomm_with_cost:,.2f}")
print(f"  总收益率: {(pnlcomm_with_cost / 100000) * 100:.2f}%")

print(f"\n差异：")
print(f"  收益率差异: {((pnl_no_cost - pnlcomm_with_cost) / 100000) * 100:.2f}%")
print(f"  交易成本占比: {(cost_with_cost / pnl_with_cost) * 100:.1f}%" if pnl_with_cost > 0 else "")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# 资金曲线对比
ax1 = axes[0]
ax1.plot(data_df.index, data_df['close'] / data_df['close'].iloc[0] * 100000, 
         label='买入持有', linewidth=1.5, alpha=0.5)
ax1.axhline(y=100000, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(y=100000 + pnl_no_cost, color='green', linestyle='--', alpha=0.5)
ax1.axhline(y=100000 + pnlcomm_with_cost, color='red', linestyle='--', alpha=0.5)

ax1.set_xlabel('日期')
ax1.set_ylabel('资金')
ax1.set_title('资金曲线对比')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 交易成本分析
ax2 = axes[1]
labels = ['无手续费', '含手续费']
returns = [(pnl_no_cost / 100000) * 100, (pnlcomm_with_cost / 100000) * 100]
colors = ['green', 'red']

bars = ax2.bar(labels, returns, color=colors, alpha=0.7)
ax2.set_ylabel('收益率 (%)')
ax2.set_title('扣费前后收益率对比')
ax2.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, ret in zip(bars, returns):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{ret:.2f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

---
## 5. 订单类型与撮合逻辑

### 5.1 常用订单类型

| 订单类型 | 说明 | 适用场景 |
|----------|------|----------|
| **市价单** | 以当前市场价格立即成交 | 快速入场/出场 |
| **限价单** | 指定价格，达到后成交 | 精确控制成本 |
| **止损单** | 价格达到阈值后触发 | 风险控制 |

### 5.2 撮合逻辑

backtrader 的撮合逻辑：

1. **市价单**：以当天的开盘价成交
2. **限价单**：如果当天的价格触及限价，则成交
3. **止损单**：如果当天的价格触及止损价，则触发

**注意**：backtrader 默认使用**当天的开盘价**撮合，这可能与实际交易有差异。

In [ ]:
class LimitOrderStrategy(bt.Strategy):
    """
    限价单策略示例
    """
    params = (
        ('fast_period', 10),
        ('slow_period', 30),
        ('limit_pct', 0.02),  # 限价单偏离 2%
    )
    
    def __init__(self):
        self.fast_ma = bt.indicators.SMA(
            self.data.close, period=self.params.fast_period
        )
        self.slow_ma = bt.indicators.SMA(
            self.data.close, period=self.params.slow_period
        )
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)
    
    def next(self):
        if not self.position:
            if self.crossover > 0:
                # 限价单：在当前价格下方 2% 买入
                limit_price = self.data.close[0] * (1 - self.params.limit_pct)
                self.buy(price=limit_price, exectype=bt.Order.Limit)
        else:
            if self.crossover < 0:
                # 限价单：在当前价格上方 2% 卖出
                limit_price = self.data.close[0] * (1 + self.params.limit_pct)
                self.sell(price=limit_price, exectype=bt.Order.Limit)


# 运行限价单策略
cerebro_limit = bt.Cerebro()
cerebro_limit.broker.setcash(100000)
cerebro_limit.broker.setcommission(commission=0.001)

data_limit = PandasData(dataname=data_df)
cerebro_limit.adddata(data_limit)
cerebro_limit.addstrategy(LimitOrderStrategy)

print("运行限价单策略...")
results_limit = cerebro_limit.run()
final_value_limit = cerebro_limit.broker.getvalue()

print(f"\n限价单策略结果：")
print(f"  初始资金: 100,000.00")
print(f"  最终资金: {final_value_limit:,.2f}")
print(f"  总收益率: {(final_value_limit / 100000 - 1) * 100:.2f}%")

---
## 6. 向量化 vs 事件驱动对比

### 6.1 向量化回测（pandas 实现）

In [ ]:
def vectorized_backtest(prices_df, fast_period=10, slow_period=30, commission=0.001):
    """
    向量化回测双均线策略
    """
    prices = prices_df['close'].copy()
    
    # 计算均线
    fast_ma = prices.rolling(window=fast_period).mean()
    slow_ma = prices.rolling(window=slow_period).mean()
    
    # 生成信号
    signals = pd.Series(0, index=prices.index)
    signals[fast_ma > slow_ma] = 1   # 金叉，买入
    signals[fast_ma < slow_ma] = -1  # 死叉，卖出
    
    # 计算仓位变化
    positions = signals.diff()
    
    # 计算收益
    returns = prices.pct_change()
    strategy_returns = signals.shift(1) * returns
    
    # 计算交易成本
    trades = positions.abs() / 2  # 交易次数（买入+卖出算一次）
    trade_costs = trades * commission
    
    # 扣费后收益
    net_returns = strategy_returns - trade_costs
    
    # 累积收益
    cumulative = (1 + net_returns).cumprod()
    
    return cumulative, net_returns, len(trades[trades > 0])


# 运行向量化回测
cumulative_vec, returns_vec, n_trades_vec = vectorized_backtest(data_df)

print(f"向量化回测结果（含手续费 0.1%）：")
print(f"  最终资金: {100000 * cumulative_vec.iloc[-1]:,.2f}")
print(f"  总收益率: {(cumulative_vec.iloc[-1] - 1) * 100:.2f}%")
print(f"  交易次数: {n_trades_vec}")

### 6.2 两种方法对比

In [ ]:
# 对比两种方法
print("=" * 60)
print("向量化 vs 事件驱动对比")
print("=" * 60)

print(f"\n向量化回测（pandas）：")
print(f"  最终资金: {100000 * cumulative_vec.iloc[-1]:,.2f}")
print(f"  总收益率: {(cumulative_vec.iloc[-1] - 1) * 100:.2f}%")

print(f"\n事件驱动回测（backtrader）：")
print(f"  最终资金: {final_value_with_cost:,.2f}")
print(f"  总收益率: {total_return_with_cost:.2f}%")

print(f"\n差异原因：")
print(f"  1. 撮合逻辑不同：backtrader 用开盘价，向量化用收盘价")
print(f"  2. 信号生成时点不同")
print(f"  3. 交易成本计算方式不同")
print(f"\n结论：事件驱动回测更接近真实交易，但速度较慢。")

---
## 7. 小结

### 核心收获

1. **事件驱动 vs 向量化**：
   - 向量化：快，但可能有未来数据泄露
   - 事件驱动：慢，但更真实

2. **backtrader 架构**：
   - Cerebro（大脑）：协调所有组件
   - DataFeed（数据源）：提供行情数据
   - Strategy（策略）：定义交易逻辑
   - Broker（经纪人）：处理订单和资金

3. **交易成本的影响**：
   - 手续费 0.1% + 滑点 0.05%
   - 频繁交易会显著降低收益

4. **订单类型**：
   - 市价单：快速成交，但价格不确定
   - 限价单：价格确定，但可能不成交

### 验收标准 Checklist

- [x] **能用两种框架实现同一策略**：向量化（pandas）和事件驱动（backtrader）
- [x] **理解交易成本的影响**：手续费 0.1% 会显著降低收益
- [x] **明白撮合逻辑的影响**：backtrader 用开盘价撮合，与向量化不同

---

**恭喜你完成了 backtrader 事件驱动回测的学习！** 🎉

你现在掌握了两种回测框架，可以根据需要选择使用。